#### **Imports**

In [26]:
import os
import httpx
from dotenv import load_dotenv
from typing import TypedDict, Literal, Optional, Annotated

from pydantic import BaseModel

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from IPython.display import Image

load_dotenv()

True

In [23]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
DATA_API_BASE_URL = "http://127.0.0.1:8000/api/v1"

#### **SKU Map**

In [4]:
SKU_MAP: dict[str, str] = {
    "FLOW_FREE": "f30db892-07e9-47e9-837c-80727f46fd3d",
    "CCIBOTS_PRIVPREV_VIRAL": "606b54a9-78d8-4298-ad8b-df6ef4481c80",
    "DEVELOPERPACK_E5": "c42b9cae-ea4f-4ab7-9717-81576235ccac",
    "POWERAPPS_VIRAL": "dcb1a3ae-b33f-4487-846a-a640262fadf4",
    "POWERAPPS_DEV": "5b631642-bd26-49fe-bd20-1daaa972ef80"
}

#### **LangGraph State**

In [17]:
class M365OperationState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_input: str
    intent: Optional[str]
    target_user_id: Optional[str]
    license_name: Optional[str]
    sku_id: Optional[str]
    operation_result: Optional[str]
    error_message: Optional[str]

#### **Router Structured Output Schema**

In [6]:
class RouterOutput(BaseModel):
    intent: Literal["assign_license", "revoke_license", "unknown"]
    target_user_id: Optional[str] = None
    license_name: Optional[str] = None
    reasoning: str

#### **Defining Tools**

In [ ]:
@tool
def assign_license(user_id: str, sku_id: str) -> str:
    """
    Assigns a M365 license to a user via the M365 Data API
    
    Args:
        user_id: Object ID of the user
        sku_id: SKU ID of the license to be assigned
    """
    try:
        res = httpx.post(
            f"{DATA_API_BASE_URL}/licenses/assign",
            json={"user_id": user_id, "sku_id": sku_id},
            timeout=300,
            verify=False
        )
        res.raise_for_status()

        return res.json()["message"]
    except Exception as e:
        return f"Assignment failed: {str(e)}"
    
@tool
def revoke_license(user_id: str, sku_id: str) -> str:
    """
    Revokes a M365 license to a user via the M365 Data API
    
    Args:
        user_id: Object ID of the user
        sku_id: SKU ID of the license to be revoked
    """
    try:
        res = httpx.post(
            f"{DATA_API_BASE_URL}/licenses/revoke",
            json={"user_id": user_id, "sku_id": sku_id},
            timeout=300,
            verify=False
        )
        res.raise_for_status()

        return res.json()["message"]
    except Exception as e:
        return f"Assignment failed: {str(e)}"

In [25]:
tools = [assign_license, revoke_license]

In [27]:
tool_node = ToolNode(tools)

#### **LLMs**

In [29]:
qwen_llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

meta_llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0
)

router_llm = qwen_llm.with_structured_output(RouterOutput)

ROUTER_SYSTEM_PROMPT = """You are an M365 license operations router.

From the user's natural language request, extract:
- intent: "assign_license" if they want to add/give a license,
    "revoke_assign" if they want to remove/revoke one,
    "unknown" if the intent is unclear
- target_user_id: the user's object ID
- license_name: the license name mentioned (for example: "E3", "E5")
- reasoning: brief explanation of your extraction 


If any field is not present in the request, return null for that field"""

action_llm = meta_llm.bind_tools(tools=tools)

ACTION_SYSTEM_PROMPT = """You are an M365 license operations executor.

You have tools to assign or revoke licenses.
Based on the intent and parameters provided, call the correct tool immediately."""

#### **Intent Router Node**

In [8]:
def intent_router_node(state: M365OperationState) -> M365OperationState:
    result: RouterOutput = router_llm.invoke([
        SystemMessage(content=ROUTER_SYSTEM_PROMPT),
        HumanMessage(content=state["user_input"])
    ])

    sku_id = None
    if result.license_name:
        sku_id = SKU_MAP.get(result.license_name.upper().strip())

    print(f"[Router] Intent: {result.intent} | User: {result.target_user_id} "
          f"| License {result.license_name} -> SKU: {sku_id}")
    
    return {
        **state,
        "intent": result.intent,
        "target_user_id": result.target_user_id,
        "license_name": result.license_name,
        "sku_id": sku_id
    }

#### **Action Agent Node**

In [30]:
def action_agent_node(state: M365OperationState) -> M365OperationState:
    if not state.get("target_user_id"):
        return {**state, "error_message": "No target user found. Provide a vald user_id."}
    if not state.get("sku_id"):
        return {**state, "error_message": (
            f"License '{state.get("license_name")}' not found. "
            f"Known licenses: {list(SKU_MAP.keys())}"
        )}
    
    prompt = (
        f"Intent: {state.get("intent")}\n"
        f"User ID: {state.get("target_user_id")}\n"
        f"SKU ID: {state.get("sku_id")}\n\n"
        f"Call the appropriate tool now."
    )

    response = action_llm.invoke([
        SystemMessage(content=ACTION_SYSTEM_PROMPT),
        HumanMessage(content=prompt)
    ])

    return {"messages": [response]}

#### **Assign License Node**

In [31]:
# def assign_license_node(state: M365OperationState) -> M365OperationState:
#     user_id = state.get("target_user_id")
#     sku_id = state.get("sku_id")

#     if not user_id:
#         return {**state, "error_message": "No target user found in request. Provide a user_id"}
#     if not sku_id:
#         return {**state, "error_message": (
#             f"{state.get("license_name")} not found in SKU map. "
#             f"Known licenses: {list(SKU_MAP.keys())}"
#         )}
    
#     print("Assign license node called")
#     return {**state, "operation_result": "success"}

#### **Revoke License Node**

In [32]:
# def revoke_license_node(state: M365OperationState) -> M365OperationState:
#     user_id = state.get("target_user_id")
#     sku_id = state.get("sku_id")

#     if not user_id:
#         return {**state, "error_message": "No target user found in request. Provide a user_id"}
#     if not sku_id:
#         return {**state, "error_message": (
#             f"{state.get("license_name")} not found in SKU map. "
#             f"Known licenses: {list(SKU_MAP.keys())}"
#         )}
    
#     print("Revoke license node called")
#     return {**state, "operation_result": "success"}

#### **Unknown Router Node**

In [33]:
def unknown_router_node(state: M365OperationState) -> M365OperationState:
    return {**state, "error_message": (
        f"Could not determine intent from '{state['user_input']}'. "
        "Specify whether you want to assign or revoke a license."
    )}

#### **Conditional Router Function**

In [34]:
def route_by_intent(state: M365OperationState) -> str:
    intent = state.get("intent", "unkown")
    
    if intent in ("assign_license", "revoke_license"):
        return "action_agent"
    
    return "unkown_intent"

#### **Build & Compile Graph**

In [35]:
builder = StateGraph(M365OperationState)

builder.add_node("intent_router", intent_router_node)
builder.add_node("action_agent", action_agent_node)
builder.add_node("tools", tool_node)
builder.add_node("unknown_intent", unknown_router_node)

builder.add_edge(START, "intent_router")
builder.add_conditional_edges(
    "intent_router",
    route_by_intent,
    {
        "action_agent": "action_agent",
        "unknown_intent": "unknown_intent" 
    }
)

builder.add_conditional_edges("action_agent", tools_condition)

builder.add_edge("tools", END)
builder.add_edge("unknown_intent", END)

graph = builder.compile()

graph.get_graph().print_ascii()

                      +-----------+                            
                      | __start__ |                            
                      +-----------+                            
                            *                                  
                            *                                  
                            *                                  
                    +---------------+                          
                    | intent_router |                          
                    +---------------+.                         
                     ..               .....                    
                   ..                      ...                 
                 ..                           .....            
       +--------------+                            ..          
       | action_agent |                             .          
       +--------------+                             .          
         ..         ..                  

In [36]:
def run_test(user_input: str):
    print(f"Input: {user_input}")
    result = graph.invoke({"user_input": user_input})

    print(f"Intent: {result.get("intent")}")
    print(f"User: {result.get("target_user_id")}")
    print(f"License: {result.get("license_name")} -> {result.get("sku_id")}")
    print(f"Result: {result.get("operation_result")}")
    print(f"Error: {result.get("error_message")}")

In [ ]:
run_test("Remove POWERAPPS_DEV license from the user b14ba680-2d12-4f72-8063-c7b20a1fb6e9")